# Rainfall Analysis and Planting Advisory

This project analyzes monthly rainfall data from five Kenyan counties using NumPy. The Machakos rainfall data is cleaned before being combined with the other counties. The analysis calculates annual rainfall totals, monthly averages, wettest months, dry-month counts, and recommended two-month planting windows.

In [1]:
import numpy as np
import csv

In [2]:
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],  # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],    # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

In [3]:
def clean_machakos(raw_values):
    valid_values = []
    invalid_months = []

    for i, value in enumerate(raw_values):
        try:
            valid_values.append(int(value.strip()))
        except ValueError:
            invalid_months.append(i)

    average_value = round(np.mean(valid_values))

    cleaned_values = []

    for i, value in enumerate(raw_values):
        try:
            cleaned_values.append(int(value.strip()))
        except ValueError:
            cleaned_values.append(average_value)

    return cleaned_values, invalid_months, average_value


machakos_clean, invalid_months, machakos_average = clean_machakos(machakos_raw)

print("Machakos valid-value average:", machakos_average)

print("Imputed months:")
for i in invalid_months:
    print(months[i], "->", machakos_clean[i], "mm")

print("Cleaned Machakos rainfall:")
print(machakos_clean)

Machakos valid-value average: 68
Imputed months:
April -> 68 mm
September -> 68 mm
Cleaned Machakos rainfall:
[48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


In [4]:
all_rows = clean_rows + [machakos_clean]

R = np.array(all_rows)

print("Rainfall array:")
print(R)

print("Shape:", R.shape)
print("Dtype:", R.dtype)

Rainfall array:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Shape: (5, 12)
Dtype: int64


## Annual Rainfall Totals

In [5]:
annual_totals = R.sum(axis=1)

print("Annual rainfall totals:")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")

Annual rainfall totals:
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm


In [6]:
monthly_averages = R.mean(axis=0)

print("Average rainfall by month:")

for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.2f} mm")

Average rainfall by month:
January: 47.00 mm
February: 47.60 mm
March: 91.00 mm
April: 164.40 mm
May: 171.20 mm
June: 67.60 mm
July: 56.60 mm
August: 58.20 mm
September: 64.20 mm
October: 67.80 mm
November: 127.40 mm
December: 80.60 mm


## Wettest Month by County

In [7]:
wettest_indices = np.argmax(R, axis=1)

wettest_months = [months[i] for i in wettest_indices]

for county, month in zip(counties, wettest_months):
    print(f"{county}: {month}")

Nairobi: April
Nakuru: April
Kisumu: April
Mombasa: May
Machakos: November


## Dry Month Analysis

In [8]:
dry_mask = R < 50

dry_month_counts = dry_mask.sum(axis=1)

print("Number of dry months per county:")

for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count}")

Number of dry months per county:
Nairobi: 4
Nakuru: 3
Kisumu: 0
Mombasa: 2
Machakos: 6


## Recommended Planting Windows

In [9]:
two_month_totals = R[:, :-1] + R[:, 1:]

best_window_indices = np.argmax(two_month_totals, axis=1)

planting_windows = [
    f"{months[i]}-{months[i + 1]}"
    for i in best_window_indices
]

print("Recommended planting windows:")

for county, window in zip(counties, planting_windows):
    print(f"{county}: {window}")

Recommended planting windows:
Nairobi: April-May
Nakuru: April-May
Kisumu: April-May
Mombasa: April-May
Machakos: November-December


In [10]:
with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)

    writer.writerow([
        "county",
        "annual_total_mm",
        "wettest_month",
        "dry_months",
        "planting_window"
    ])

    for county, total, wettest, dry, window in zip(
        counties,
        annual_totals,
        wettest_months,
        dry_month_counts,
        planting_windows
    ):
        writer.writerow([
            county,
            int(total),
            wettest,
            int(dry),
            window
        ])

print("rainfall_summary.csv created successfully.")

rainfall_summary.csv created successfully.


In [11]:
with open("rainfall_summary.csv", "r", encoding="utf-8") as file:
    print(file.read())

county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,April-May
Nakuru,916,April,3,April-May
Kisumu,1371,April,0,April-May
Mombasa,1075,May,2,April-May
Machakos,811,November,6,November-December



## Planting Advisory

The NGO should use the two-month rainfall windows identified in this analysis as a guide when sending planting SMS advice to farmers. These windows are based on the highest combined rainfall across consecutive months for each county and may indicate periods with better rainfall availability for planting. Farmers should also consider local soil conditions, crop requirements, and short-term weather forecasts before planting. Rainfall patterns can vary within counties, so station-level results may not represent every farming area. The Machakos data required cleaning because two monthly values were invalid and were replaced using the rounded average of the valid observations. The NGO should therefore treat the results as advisory information rather than a guarantee of future rainfall.

## Data Quality Notes

The Machakos station data contained two invalid entries: "N/A" for April and a blank value for September. These values were replaced with the rounded mean of the valid Machakos rainfall observations. The analysis assumes that the remaining values are accurate and that the station readings are representative of each county. The results should therefore be interpreted with awareness of possible measurement and data-entry limitations.